# Kaggle Titanic: Other ML Models (ExtraTrees, GradientBoosting, SVC, KNN)

In [ ]:
# 1) Imports and Paths
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import accuracy_score

from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Paths and constants
DATA_DIR = "/home/atul-kumar/workspace/kaggle/titanic/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_STATE = 63
np.random.seed(RANDOM_STATE)
print("Paths set:", TRAIN_PATH, TEST_PATH)

Paths set: /home/atul-kumar/workspace/kaggle/titanic/data/train.csv /home/atul-kumar/workspace/kaggle/titanic/data/test.csv


In [2]:
# 2) Load Data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head(3)

Train shape: (891, 12)  Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [ ]:
# 3) Shared Feature Engineering (consistent with other notebooks)
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Title + bucket
    out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.")
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Dona': 'Rare', 'Sir': 'Rare', 'Don': 'Rare',
        'Jonkheer': 'Rare', 'Capt': 'Rare', 'Col': 'Rare', 'Dr': 'Rare', 'Rev': 'Rare',
        'Major': 'Rare'
    }
    out["TitleBucket"] = out["Title"].replace(title_map)
    out.loc[~out["TitleBucket"].isin(['Mr','Mrs','Miss','Master','Rare']), 'TitleBucket'] = 'Rare'
    # Family
    out["FamilySize"] = out.get("SibSp", 0) + out.get("Parch", 0) + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    def _family_bin(n):
        if n == 1: return 'Single'
        if 2 <= n <= 4: return 'Small'
        return 'Large'
    out["FamilySizeBin"] = out["FamilySize"].apply(_family_bin)
    # Ticket
    if 'Ticket' in out.columns:
        counts = out['Ticket'].value_counts()
        out['TicketGroup'] = out['Ticket'].map(counts)
    else:
        out['TicketGroup'] = 1
    # Cabin presence + Fare/Age transforms
    out['CabinKnown'] = out['Cabin'].notna().astype(int) if 'Cabin' in out.columns else 0
    out['FareLog'] = np.log1p(out['Fare']) if 'Fare' in out.columns else 0.0
    out['AgePclass'] = out.get('Age', np.nan) * out.get('Pclass', np.nan)
    out['FarePerPerson'] = out['Fare'] / out['FamilySize']
    return out

train_fe = add_engineered_features(train_df)
test_fe = add_engineered_features(test_df)
base_features = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']
engineered = ['TitleBucket','FamilySize','IsAlone','FamilySizeBin','TicketGroup','CabinKnown','FareLog','AgePclass','FarePerPerson']
all_features = base_features + engineered
print('Total features:', len(all_features))

Total features: 16


In [ ]:
# 4) Preprocessing blocks
numeric = ['Age','SibSp','Parch','Fare','FamilySize','TicketGroup','FareLog','AgePclass','FarePerPerson']
categorical = ['Pclass','Sex','Embarked','TitleBucket','FamilySizeBin','CabinKnown']

num_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))])
cat_tf_ohe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
cat_tf_scale = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)), 
    ('scaler', StandardScaler(with_mean=False))])

pre_ohe = ColumnTransformer([
    ('num', num_tf, numeric),
    ('cat', cat_tf_ohe, categorical)])
pre_scale = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric),
    ('cat', cat_tf_scale, categorical)])

X = train_fe[all_features]; y = train_fe['Survived']
X_test = test_fe[all_features]

print('Shapes:', X.shape, X_test.shape)

Shapes: (891, 16) (418, 16)


In [ ]:
# 5) ExtraTrees (robust to noise, fast)
et = Pipeline([
    ('pre', pre_ohe),
    ('clf', ExtraTreesClassifier(random_state=RANDOM_STATE))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
et_grid = {
  'clf__n_estimators': [300, 600, 900],
  'clf__max_depth': [None, 6, 8, 12],
  'clf__min_samples_leaf': [1, 2, 3],
  'clf__min_samples_split': [2, 4, 6],
  'clf__max_features': [0.4, 0.6, 'sqrt'],
  'clf__bootstrap': [False],
}
et_search = RandomizedSearchCV(
    et, 
    et_grid, 
    n_iter=30, 
    scoring='accuracy', 
    cv=cv, 
    n_jobs=-1, 
    random_state=RANDOM_STATE, 
    refit=True, 
    verbose=1)
et_search.fit(X, y)
print('ET best params:', et_search.best_params_)
print('ET CV Accuracy:', round(et_search.best_score_, 4))

p_oof = cross_val_predict(
    et_search.best_estimator_,
    X,
    y,
    cv=cv,
    method='predict_proba',
    n_jobs=-1)[:,1]
best_t, best_acc = 0.5, -1
for t in np.linspace(0.35, 0.65, 61):
    a = accuracy_score(y, (p_oof >= t).astype(int))
    if a > best_acc: best_acc, best_t = a, float(t)
print(f'ET best threshold {best_t:.3f} ACC {best_acc:.4f}')

Fitting 5 folds for each of 30 candidates, totalling 150 fits
ET best params: {'clf__n_estimators': 600, 'clf__min_samples_split': 6, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': 12, 'clf__bootstrap': False}
ET CV Accuracy: 0.8294
ET best params: {'clf__n_estimators': 600, 'clf__min_samples_split': 6, 'clf__min_samples_leaf': 1, 'clf__max_features': 'sqrt', 'clf__max_depth': 12, 'clf__bootstrap': False}
ET CV Accuracy: 0.8294
ET best threshold 0.545 ACC 0.8316
ET best threshold 0.545 ACC 0.8316


In [ ]:
# 6) GradientBoosting (tree-based, no hist bins)
gb = Pipeline([
    ('pre', pre_ohe),
    ('clf', GradientBoostingClassifier(random_state=RANDOM_STATE))])
gb_grid = {
  'clf__n_estimators': [150, 250, 350],
  'clf__learning_rate': [0.05, 0.08, 0.1],
  'clf__max_depth': [2, 3],
  'clf__min_samples_leaf': [1, 2, 3],
  'clf__subsample': [0.7, 0.85, 1.0],
}
gb_search = RandomizedSearchCV(
    gb,
    gb_grid,
    n_iter=30,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1)
gb_search.fit(X, y)
print('GB best params:', gb_search.best_params_)
print('GB CV Accuracy:', round(gb_search.best_score_, 4))

p_oof = cross_val_predict(
    gb_search.best_estimator_,
    X,
    y,
    cv=cv,
    method='predict_proba',
    n_jobs=-1)[:,1]
best_t, best_acc = 0.5, -1
for t in np.linspace(0.35, 0.65, 61):
    a = accuracy_score(y, (p_oof >= t).astype(int))
    if a > best_acc: best_acc, best_t = a, float(t)
print(f'GB best threshold {best_t:.3f} ACC {best_acc:.4f}')

Fitting 5 folds for each of 30 candidates, totalling 150 fits
GB best params: {'clf__subsample': 0.7, 'clf__n_estimators': 150, 'clf__min_samples_leaf': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05}
GB CV Accuracy: 0.835
GB best threshold 0.525 ACC 0.8384
GB best params: {'clf__subsample': 0.7, 'clf__n_estimators': 150, 'clf__min_samples_leaf': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05}
GB CV Accuracy: 0.835
GB best threshold 0.525 ACC 0.8384


In [ ]:
# 7) SVC (RBF) with probability + scaling
svc = Pipeline([
    ('pre', pre_scale),
    ('clf', SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE))])
svc_grid = {
  'clf__C': [0.5, 1.0, 2.0, 4.0],
  'clf__gamma': ['scale', 0.1, 0.05, 0.02],
}
svc_search = RandomizedSearchCV(
    svc, 
    svc_grid, 
    n_iter=16,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True, verbose=1)
svc_search.fit(X, y)
print('SVC best params:', svc_search.best_params_)
print('SVC CV Accuracy:', round(svc_search.best_score_, 4))

p_oof = cross_val_predict(
    svc_search.best_estimator_,
    X,
    y,
    cv=cv,
    method='predict_proba',
    n_jobs=-1)[:,1]
best_t, best_acc = 0.5, -1
for t in np.linspace(0.35, 0.65, 61):
    a = accuracy_score(y, (p_oof >= t).astype(int))
    if a > best_acc: best_acc, best_t = a, float(t)
print(f'SVC best threshold {best_t:.3f} ACC {best_acc:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits
SVC best params: {'clf__gamma': 'scale', 'clf__C': 2.0}
SVC CV Accuracy: 0.8395
SVC best threshold 0.520 ACC 0.8406
SVC best params: {'clf__gamma': 'scale', 'clf__C': 2.0}
SVC CV Accuracy: 0.8395
SVC best threshold 0.520 ACC 0.8406


In [ ]:
# 8) KNN with scaling
knn = Pipeline([
    ('pre', pre_scale),
    ('clf', KNeighborsClassifier())])
knn_grid = {
  'clf__n_neighbors': [5, 7, 9, 11, 15],
  'clf__weights': ['uniform', 'distance'],
  'clf__p': [1, 2],
}
knn_search = RandomizedSearchCV(
    knn,
    knn_grid,
    n_iter=16,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
    verbose=1)
knn_search.fit(X, y)
print('KNN best params:', knn_search.best_params_)
print('KNN CV Accuracy:', round(knn_search.best_score_, 4))

p_oof = cross_val_predict(knn_search.best_estimator_, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:,1]
best_t, best_acc = 0.5, -1
for t in np.linspace(0.35, 0.65, 61):
    a = accuracy_score(y, (p_oof >= t).astype(int))
    if a > best_acc: best_acc, best_t = a, float(t)
print(f'KNN best threshold {best_t:.3f} ACC {best_acc:.4f}')

Fitting 5 folds for each of 16 candidates, totalling 80 fits
KNN best params: {'clf__weights': 'uniform', 'clf__p': 2, 'clf__n_neighbors': 15}
KNN CV Accuracy: 0.8283
KNN best threshold 0.535 ACC 0.8316


In [9]:
# 9) Predict test and save submissions for each
models = {
  'extratrees': et_search.best_estimator_,
  'gradboost': gb_search.best_estimator_,
  'svc': svc_search.best_estimator_,
  'knn': knn_search.best_estimator_,
}
for name, mdl in models.items():
    mdl.fit(X, y)
    p = mdl.predict_proba(X_test)[:,1]
    # reuse a central threshold (0.5); you can swap in each best_t if stored individually
    labels = (p >= 0.5).astype(int)
    path = os.path.join(DATA_DIR, f'submission-{name}.csv')
    pd.DataFrame({'PassengerId': test_fe['PassengerId'], 'Survived': labels}).to_csv(path, index=False)
    print('Saved', name, 'submission to:', path)
print('Done')

Saved extratrees submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-extratrees.csv
Saved gradboost submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-gradboost.csv
Saved svc submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-svc.csv
Saved knn submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-knn.csv
Done
Saved knn submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission-knn.csv
Done
